# Deep_Monocular_VO — Colab training templateCopy this notebook, change `MODEL` below, and run top to bottom.**Read this once.** Colab's disk is wiped when the session ends, so:- the **dataset** is extracted from Drive to Colab's local disk (fast I/O, disposable)- **checkpoints** are written straight to Drive, so a disconnect costs you one epoch, not the run- **code** lives in git, never only in this notebookNever edit model code in a Colab cell. Edit it locally, commit, push, and re-clone here —otherwise the work is not in your commit history, which is what the project is graded on.

## 1. Confirm we actually have a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csvimport torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())assert torch.cuda.is_available(), (    "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), then re-run.")

## 2. Mount Drive and unpack the dataset`vkitti2_subset.tar.gz` (1.5 GB) is the shared archive — do **not** re-download the 7 GBoriginal. Extracting to `/content` rather than reading from Drive matters: Drive I/O is slowenough to bottleneck the dataloader.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DRIVE = '/content/drive/MyDrive/DL_PROJECT'      # <- adjust to your shared folderDATA  = '/content/data'import os, pathlibos.makedirs(DATA, exist_ok=True)if not pathlib.Path(f'{DATA}/vkitti2/Scene20/clone/extrinsic.txt').exists():    !tar xzf "{DRIVE}/vkitti2_subset.tar.gz" -C {DATA}else:    print('dataset already extracted')!find {DATA}/vkitti2 -name '*.jpg' | wc -l          # expect 8642

## 3. Clone the repoThe repo is private, so cloning needs a token. **Do not paste it into a cell** — a tokencommitted to git is a token you have to revoke.Use Colab's Secrets instead (🔑 icon in the left sidebar): add a secret named `GH_TOKEN`holding a fine-grained PAT with *Contents: read* on this repo, and enable notebook access.

In [ ]:
from google.colab import userdataTOKEN = userdata.get('GH_TOKEN')REPO = 'VaibhavBansal05/Deep_Monocular_VO'BRANCH = 'main'                                   # or your feature branch%cd /content!rm -rf Deep_Monocular_VO!git clone --quiet --branch {BRANCH} https://{TOKEN}@github.com/{REPO}.git%cd /content/Deep_Monocular_VO/src!git log --oneline -1

In [ ]:
!pip install -q -e ".[dev]"!pytest -q                                        # contract tests must pass

## 4. TrainCheckpoints go to Drive so a disconnect is survivable. If the session drops, re-runeverything above and add `--resume` to pick up from the last epoch.

In [ ]:
MODEL = 'm4'                                      # m1 | m2 | m3 | m4OUT   = f'{DRIVE}/checkpoints/{MODEL}'!mkdir -p "{OUT}"!python scripts/train.py --model {MODEL} \    --config configs/{MODEL}.yaml \    --data-root {DATA}/vkitti2 \    --out "{OUT}"# after a disconnect, instead run:# !python scripts/train.py --model {MODEL} --data-root {DATA}/vkitti2 \#     --out "{OUT}" --resume "{OUT}/last.pt"

## 5. Evaluate and bring the results home

In [ ]:
!python scripts/eval.py --model {MODEL} \    --ckpt "{OUT}/best.pt" \    --data-root {DATA}/vkitti2!cp -r results/{MODEL} "{DRIVE}/results_{MODEL}"!cat results/{MODEL}/metrics.json

## 6. Commit the results — from your laptop, not here`results/<model>/metrics.json`, the trajectory `.txt` and the plots are small, and theyare the evidence trail for the Interim marks. Copy them out of Drive locally and committhem on your branch:```bashcp -r ~/Drive/DL_PROJECT/results_m4/* src/results/m4/git add src/results/m4 && git commit -m "Add M4 test results"```Committing from Colab is possible but needs a write-scoped token in the session, which isa credential-leak risk for no real benefit. Commit locally.**Never commit** `*.pt` checkpoints — they stay in Drive.---### Colab vs KaggleFree Colab gives a T4 and disconnects when idle; fine for M1/M4. Kaggle gives P100 or 2×T4with a 30 h/week quota and tolerates long unattended runs better — worth it for M3, whichtrains longest. The setup is identical apart from mounting Kaggle datasets instead of Drive.